# 每月调仓决策助手

**策略**: U11 + residual_mom + n=4 + rank weighting

**操作流程**（每月第一个交易日做一次）：

1. 在下面的 Inputs cell 里填写**当前持仓**（IBKR 账户里实际持有的 ETF 和股数）+ **可用现金余额**
2. 点 Cell → Run All（或 ⇧⏎ 一格一格执行）
3. 最后一个 cell 输出**交易清单**：买卖什么、多少股
4. 去 IBKR 下单（市价单 OK，monthly rebalance 不在乎几个 bp 的滑点）

**信号源**：yfinance 实时数据（免费）。  
**理论依据**：见 `docs/research_plan.md` 和 `reports/strategy_backtest_report.md`。  
**Backtest 表现** (2010-2026, 16 年, U11+rank+n=4): CAGR 14.3%, Sharpe 1.20, MDD -15.6%。

## 1. Imports + 配置（不要改）

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Strategy parameters (locked in after diagnostic tests)
UNIVERSE = ['SPY', 'QQQ', 'EFA', 'TLT', 'GLD',
            'VWO', 'HYG', 'VNQ', 'IEF', 'DBC', 'SOXX']
LONG_N   = 4                  # 持有数量
SCHEME   = 'rank'             # 权重方案: rank = 40/30/20/10

# Rank weights for n=4 (top → bottom): 40% / 30% / 20% / 10%
RANK_WEIGHTS_N4 = [4/10, 3/10, 2/10, 1/10]

print(f'Strategy: U{len(UNIVERSE)} + residual_mom + n={LONG_N} + {SCHEME} weighting')
print(f'Universe: {UNIVERSE}')

## 2. ★ Inputs ★（**每月只改这一格**）

把 IBKR 账户里**今天**的实际状态填进去：

In [ ]:
# 当前持仓：ticker -> 持有股数（整数）。
# 没持有的 ETF 不要列；首次运行就留空 dict {}
current_holdings = {
    # 'SPY': 12,
    # 'TLT': 25,
    # 'GLD': 18,
    # 'SOXX': 5,
}

# 当前可用现金余额（USD）。包含本月刚 deposit 进去的 $3k。
cash_balance = 23000.0     # 例：$20k 起始 + $3k 月度 deposit

# As-of date：通常设为今天。可以指定历史日期做回溯演练。
# 留 None 即用 today。
as_of_date = None          # 例如: '2026-06-02'

# ─── 不需要改的派生变量 ───
if as_of_date is None:
    as_of_date = pd.Timestamp.today().normalize()
else:
    as_of_date = pd.Timestamp(as_of_date)
print(f'As of: {as_of_date.date()}')
print(f'Cash:  ${cash_balance:,.2f}')
print(f'Current holdings: {dict(current_holdings) or "(empty)"}')

## 3. 拉数据（自动）

In [ ]:
# Need 252+30 trading days of history for the 12-1 momentum + beta regression.
# Fetch 400 calendar days to be safe.
fetch_start = (as_of_date - timedelta(days=400)).strftime('%Y-%m-%d')
fetch_end   = (as_of_date + timedelta(days=1)).strftime('%Y-%m-%d')

raw = yf.download(UNIVERSE, start=fetch_start, end=fetch_end,
                  progress=False, auto_adjust=True)
closes = raw['Close'].dropna(how='all')

print(f'Data fetched: {len(closes)} trading days from '
      f'{closes.index.min().date()} to {closes.index.max().date()}')

# Latest closes (used for converting target $ -> target shares)
latest_prices = closes.iloc[-1]
print(f'\nLatest prices (as of {closes.index[-1].date()}):')
for t in UNIVERSE:
    p = latest_prices.get(t)
    print(f'  {t:>4}: ${p:>8.2f}' if pd.notna(p) else f'  {t:>4}: MISSING')

## 4. 计算信号（residual_mom）

对每个 ETF：剥除其相对 SPY 的 beta，残差累积回报作为信号。**这就是 `3d_xsmom_lo` 里 `signal=residual_mom` 的逻辑**。

In [ ]:
def residual_mom_latest(asset_px: pd.Series, market_px: pd.Series) -> float:
    """Compute residual_mom for the latest date.
    Uses prices from t-252 to t-21 (231-day window ending 1 month ago)."""
    asset_ret = asset_px.pct_change()
    mkt_ret = market_px.pct_change()
    aligned = pd.concat([asset_ret, mkt_ret], axis=1, keys=['a', 'm']).dropna()
    if len(aligned) < 252:
        return np.nan
    win = aligned.iloc[-252:-21]
    if len(win) < 100:
        return np.nan
    a, m = win['a'].values, win['m'].values
    cov = float(np.cov(a, m, ddof=0)[0, 1])
    var = float(np.var(m, ddof=0))
    beta = cov / var if var > 1e-12 else 1.0
    return float((a - beta * m).sum())

signals = {}
for t in UNIVERSE:
    if t not in closes.columns or closes[t].isna().all():
        print(f'  WARN: {t} no data, skipping')
        continue
    s = residual_mom_latest(closes[t], closes['SPY'])
    signals[t] = s

sig_df = pd.DataFrame({
    'signal':     [signals[t] for t in UNIVERSE],
    'rank':       0,  # filled below
    'positive':   [signals[t] > 0 for t in UNIVERSE],
    'last_price': [latest_prices.get(t) for t in UNIVERSE],
}, index=UNIVERSE)
sig_df = sig_df.sort_values('signal', ascending=False)
sig_df['rank'] = range(1, len(sig_df) + 1)

print(f'\nSignal table (sorted descending):')
with pd.option_context('display.float_format', '{:+.4f}'.format):
    print(sig_df.to_string())

## 5. 选 top 4 + 分配仓位

In [ ]:
# 只取正动量 top 4
positives = sig_df[sig_df['positive']].head(LONG_N)
k = len(positives)

if k == 0:
    print('⚠️  没有正动量 ETF — 全部持现金。本月不需要交易（如果当前已经是现金）。')
    targets_pct = {}
elif k < LONG_N:
    print(f'⚠️  只有 {k}/{LONG_N} 个 ETF 正动量。会有 {(1-k/LONG_N)*100:.0f}% 仓位留作现金。')
    # rank weights scaled to deployed = k/LONG_N
    weights_top4 = [(LONG_N - i) for i in range(LONG_N)]   # [4, 3, 2, 1]
    raw = weights_top4[:k]   # 取前 k 个
    sum_all = sum(weights_top4)
    targets_pct = {positives.index[i]: raw[i] / sum_all for i in range(k)}
else:
    # 完整 4 个 winners，40/30/20/10
    targets_pct = {positives.index[i]: RANK_WEIGHTS_N4[i] for i in range(LONG_N)}

# 计算总组合价值（现金 + 现有持仓市值）
holdings_value = sum(shares * latest_prices.get(t, 0)
                     for t, shares in current_holdings.items()
                     if t in latest_prices and pd.notna(latest_prices.get(t)))
total_value = cash_balance + holdings_value
print(f'\n当前组合：')
print(f'  持仓市值 ${holdings_value:,.2f} + 现金 ${cash_balance:,.2f} = 总值 ${total_value:,.2f}')

print(f'\n目标分配（rank weighting）：')
for t, w in targets_pct.items():
    target_dollars = total_value * w
    target_shares  = int(round(target_dollars / latest_prices[t]))
    print(f'  {t:>4}  rank #{positives.index.get_loc(t)+1}  '
          f'weight {w*100:>5.1f}%  → ${target_dollars:>10,.0f} ≈ {target_shares} shares @ ${latest_prices[t]:.2f}')

## 6. 交易清单（**复制到 IBKR 执行**）

In [ ]:
# 计算 target shares per ticker
target_shares = {}
for t in UNIVERSE:
    if t in targets_pct:
        target_shares[t] = int(round(total_value * targets_pct[t] / latest_prices[t]))
    else:
        target_shares[t] = 0

# Diff: target - current
trade_rows = []
for t in UNIVERSE:
    curr = current_holdings.get(t, 0)
    tgt = target_shares.get(t, 0)
    diff = tgt - curr
    if diff == 0 and curr == 0:
        continue  # nothing to show
    if pd.isna(latest_prices.get(t)):
        continue
    action = 'HOLD' if diff == 0 else ('BUY' if diff > 0 else 'SELL')
    trade_rows.append({
        'action':  action,
        'ticker':  t,
        'current': curr,
        'target':  tgt,
        'delta':   diff,
        'price':   latest_prices[t],
        'amount':  abs(diff) * latest_prices[t],
    })

trade_df = pd.DataFrame(trade_rows)
if trade_df.empty:
    print('✅ 当前持仓已与目标一致，本月无需交易。')
else:
    trade_df = trade_df.sort_values(['action', 'amount'], ascending=[True, False])
    # Print in execution order: SELLs first (free up cash), then BUYs
    sells = trade_df[trade_df.action == 'SELL']
    buys  = trade_df[trade_df.action == 'BUY']
    holds = trade_df[trade_df.action == 'HOLD']

    print('═' * 60)
    print(f'   交易清单（{as_of_date.date()}）')
    print('═' * 60)
    if not sells.empty:
        print('\n→ 先 SELL（先卖出腾现金）:')
        for _, r in sells.iterrows():
            print(f"   SELL  {r['ticker']:>4}  {r['delta']:>+5d} shares  @ ~${r['price']:.2f}  = ${r['amount']:>10,.0f}")
    if not buys.empty:
        print('\n→ 再 BUY:')
        for _, r in buys.iterrows():
            print(f"   BUY   {r['ticker']:>4}  {r['delta']:>+5d} shares  @ ~${r['price']:.2f}  = ${r['amount']:>10,.0f}")
    if not holds.empty:
        print('\n→ HOLD（不动）:')
        for _, r in holds.iterrows():
            print(f"   HOLD  {r['ticker']:>4}  {r['current']:>5d} shares  (no change)")

    total_sell = sells['amount'].sum() if not sells.empty else 0
    total_buy  = buys['amount'].sum()  if not buys.empty  else 0
    print(f"\n💵 净现金变动: SELL ${total_sell:,.0f} - BUY ${total_buy:,.0f} "
          f"= ${total_sell - total_buy:+,.0f}")
    print(f"💰 交易完成后预计现金余额: ${cash_balance + total_sell - total_buy:,.0f}")
    print('═' * 60)

## 7. 健康检查 + 备忘

执行 IBKR 下单时：
- 用 **market order**，monthly rebalance 不在乎 1-2 bp 滑点
- 先卖后买（先腾出现金）
- 整数股，零碎现金留作下月 buffer
- 执行完后**记录当时的持仓数字**，下个月填进 Inputs cell

**异常情况**：
- 如果某月 signal 全部 ≤ 0（全资产负动量）→ 全部清仓持现金。这是策略的避险机制，不要质疑。
- 如果连续 3+ 个月持仓完全一致 → 信号稳定，正常。
- 如果某月 turnover > 60%（一半以上仓位换掉）→ 检查是不是 signal 计算异常。

**调仓时间**：每月**第一个交易日** 10:00 ET 后跑 notebook（确保前一日数据已 settle）。如果错过几天问题不大。

**Deposit 节奏**：下次 $3k 到账后，更新 `cash_balance`，下月调仓自动会把新现金 deploy 进去。

In [ ]:
# 把本次决策结果保存到文本文件（可选，便于事后审计）
import os
log_path = f'rebalance_log_{as_of_date.date()}.txt'
log_full_path = os.path.join(os.path.dirname(os.path.abspath('__file__')) if False else '.', log_path)

with open(log_path, 'w') as f:
    f.write(f'Rebalance decision @ {as_of_date.date()}\n')
    f.write(f'Strategy: U{len(UNIVERSE)} + residual_mom + n={LONG_N} + {SCHEME}\n')
    f.write(f'\nSignal table:\n{sig_df.to_string()}\n')
    f.write(f'\nCurrent holdings: {current_holdings}\n')
    f.write(f'Cash balance: ${cash_balance:,.2f}\n')
    f.write(f'Total portfolio value: ${total_value:,.2f}\n')
    f.write(f'\nTarget weights:\n')
    for t, w in targets_pct.items():
        f.write(f'  {t}: {w*100:.1f}%  ({target_shares[t]} shares)\n')
    if not trade_df.empty:
        f.write(f'\nTrades:\n{trade_df.to_string()}\n')

print(f'✅ 决策记录已保存到 {log_path}')